# Model notebook template (CSCI167 Rotten Tomatoes project)

Make **your own copy** of this notebook for your architecture (File > Save a copy in Drive, or copy the file
in the repo, e.g. `notebooks/<yourname>_<model>.ipynb`).

Everything in here is **shared code that all of us use**: setup, data and splits, the common training settings,
the run function, the results table, learning curves and error analysis. Do not edit `common.py` or the shared
settings, so that all models are trained and evaluated the same way and the results are comparable.
The only part you write is the architecture, in the cell(s) under **YOUR ARCHITECTURE**.

Colab: Runtime > Change runtime type > GPU (T4), then run cells top to bottom.

In [ ]:
%pip install -q datasets transformers
import os
if not os.path.exists("common.py"):
    REPO_URL = "https://github.com/ZhongVang/csci167-rotten-tomatoes.git"   # group repo (private until submission)
    !git clone $REPO_URL project
    %cd project

In [ ]:
# Optional (Colab): keep results on Drive so they survive session timeouts.
# Must run BEFORE `import common`.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ["RESULTS_DIR"] = "/content/drive/MyDrive/CSCI167_results"
except Exception:
    pass  # running locally: results go to ./results

In [ ]:
OWNER = "yourname"                   # TODO: your name; results go to results_<OWNER>.csv
RESULTS_FILE = f"results_{OWNER}.csv"
SEEDS = [0, 1, 2]                    # shared: do not change

import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from common import *

print(save_env_info(OWNER))          # package versions + hardware, for the report
data = load_data()
print({k: len(v[0]) for k, v in data.items()}, "-> train/val/test sizes")
print("label counts (train):", np.bincount(data["train"][1]))
print("example:", data["train"][0][0], "->", data["train"][1][0])

## Shared training settings (do not change for the main runs)

Every model in the group is trained with the same budget so the comparison is fair: Adam, lr 1e-3, batch 64,
at most 20 epochs, early stopping (patience 5) on **validation** accuracy; the best-validation weights are
restored and the test set is evaluated once at the end. The same 3 seeds are used for 100%, 25% and 10% of the
training data. Data: official splits of `cornell-movie-review-data/rotten_tomatoes`
(train 8,530 / validation 1,066 / test 1,066).

In [ ]:
TAG = "mymodel"               # TODO: short unique name used in run names / files (e.g. "cnn")
MODEL_NAME = "My model"       # TODO: descriptive name shown in the results table

PROTOCOL = dict(optimizer="adam", lr=1e-3, batch_size=64, epochs=20, patience=5)
FRACS = [1.0, 0.25, 0.1]

def run(seed, frac, tag=None, **overrides):
    """Train + evaluate one run and log it to results/<RESULTS_FILE>.
    overrides: training settings (optimizer, lr, weight_decay, schedule, momentum) go to the trainer;
    anything else is passed to MyModel(...) as a model hyperparameter (e.g. dropout=0.0)."""
    train_keys = {"optimizer", "lr", "weight_decay", "schedule", "momentum"}
    tcfg = {**PROTOCOL, **{k: v for k, v in overrides.items() if k in train_keys}}
    mcfg = {k: v for k, v in overrides.items() if k not in train_keys}
    set_seed(seed)                                    # BEFORE building the model
    train_dl, val_dl, test_dl, vocab = make_vocab_loaders(data, tcfg["batch_size"], seed, train_frac=frac)
    model = MyModel(len(vocab), **mcfg)
    return train_and_eval(model, (train_dl, val_dl, test_dl), run_name=f"{tag or TAG}_f{frac}_s{seed}",
                          model_name=MODEL_NAME, seed=seed, train_frac=frac, extra=mcfg,
                          results_file=RESULTS_FILE, **tcfg)

## YOUR ARCHITECTURE

The architectures have not been chosen yet. Add your own code cell(s) below this one and define your model there.
You can build any deep learning architecture suited to text classification, as long as it is different from the
other group members' models. Some starting points (not a requirement): a fully connected network on pooled word
embeddings, a 1D CNN, an RNN/LSTM, a transformer, or a pretrained model (`HFClassifier` in `common.py` wraps a
Hugging Face model, fine-tuned or with a frozen encoder). The course textbook (Understanding Deep Learning)
covers fully connected networks (Ch 3-4), convolutional networks (Ch 10), residual networks (Ch 11) and
transformers (Ch 12).

**What the shared code expects:** a class named `MyModel(nn.Module)` with

- `__init__(self, vocab_size, **your_hyperparameters)`: every extra argument needs a default value, so
  `MyModel(vocab_size)` works. Index 0 is the padding token.
- `forward(self, ids, mask)`: `ids` is a LongTensor `[batch, length]` of token indices, `mask` is a BoolTensor
  `[batch, length]` that is True for real tokens and False for padding. It must return logits of shape
  `[batch, 2]` (class 0 = negative, 1 = positive).

(Pretrained Hugging Face models use their own tokenizer, so they need `make_hf_loaders` and a separate run
function. Ask the group before choosing one, so the comparison stays consistent.)

**Write down for the report:** why you chose this architecture, its structure and trainable parameter count
(printed below), how its depth and layer choices might affect learning, and what is trained or frozen if you
use a pretrained model or library code (cite the source).

**Hyperparameter study (the rubric requires one; the group decides who runs it):** investigate at least two
training factors with at least two settings each, changing one factor at a time versus a reference config, and
choose settings using validation accuracy only. Possible factors: optimizer, learning rate, batch size,
lr schedule, regularization (dropout, weight decay), augmentation. You can run each setting with the shared
function, for example `run(seed, 1.0, tag=TAG + "_sgd", optimizer="sgd", lr=0.05)` for each seed in `SEEDS`,
and compare them in the summary table below (convergence speed, validation accuracy, stability, training time).

## Sanity check (run once your model is defined)

Builds your model, prints its structure and trainable parameter count (needed for the report), and checks
the output shape on a real batch. Then run one short test (10% of the data) before the full runs.

In [ ]:
set_seed(0)
train_dl, val_dl, test_dl, vocab = make_vocab_loaders(data, 64, 0)
model = MyModel(len(vocab))
print(model)
print("trainable / total params:", count_params(model))
inputs, y = next(iter(train_dl))
out = model(**inputs)
assert out.shape == (len(y), 2), f"forward() must return logits of shape [batch, 2], got {tuple(out.shape)}"
print("OK: logits", tuple(out.shape), "| ids", tuple(inputs['ids'].shape))

In [ ]:
run(0, 0.1, tag=TAG + '_test')   # quick test: if validation accuracy is above 50%, launch the full runs

## Main runs: 3 seeds x 100% / 25% / 10% of the training data

In [ ]:
for frac in FRACS:
    for seed in SEEDS:
        run(seed, frac)

## Results: summary table, learning curves, error analysis

In [ ]:
summarize(f'results_{OWNER}.csv')   # mean +/- std over seeds, per configuration

In [ ]:
plot_history(f"{TAG}_f1.0_s0", f"{MODEL_NAME}: train vs validation (seed 0)");

In [ ]:
import json
h = json.load(open(f"{RESULTS_DIR}/history/{TAG}_f1.0_s0.json"))
preds, labels = np.array(h["test_preds"]), np.array(data["test"][1])
texts = data["test"][0]
cm = np.zeros((2, 2), int)
for y, p in zip(labels, preds): cm[y, p] += 1
print("confusion matrix (rows=true neg/pos, cols=pred neg/pos):\n", cm)
wrong = np.where(preds != labels)[0]
print(len(wrong), "errors. Examples (true, pred, text):")
for i in wrong[:20]: print(labels[i], preds[i], "|", texts[i])

## Notes for the report (fill in)

- **Design choices:** why this architecture, why these sizes/layers? What did you expect it to do well or badly?
- **Learning curves:** does it overfit (train accuracy far above validation)? When does validation peak?
- **Errors:** what do the misclassified sentences have in common (negation, sarcasm, mixed opinions, very short text)?
- **Less data:** how much does accuracy drop at 25% and 10%?
- **Limitations:** 3 seeds, small test set (about 1,066 sentences), anything you would do differently.